Upload your file to your Google Drive > Right-click > Share > Share > Change "Restricted" to "Anyone with the link > Copy link.

To speed things up, you can extract the audio and then provide the audio. You can use VLC Media Player to do so: Media > Convert/Save > Add > Choose your video > Convert/Save > Profile: Audio-MP3 > Browse > Set output file location and name (e.g. input.mp3) > Start.

Run Step 1 to load the model. Then run Step 2 and paste the link to your file (e.g., https://drive.google.com/file/d/FILE_ID/view?usp=sharing), hit Enter.

In [ ]:
!git clone https://github.com/serkansulun/autosubtitle
%cd autosubtitle
!rm -f auto_subtitle.ipynb
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# @title Step 0 - Translation settings (optional)
from ipywidgets import Dropdown, VBox, Layout
from auto_subtitle import LANGUAGE_CODES

lang_options = ["None"] + list(LANGUAGE_CODES.keys())

style_layout = Layout(width='300px')

src_lang_dropdown = Dropdown(
    options=lang_options,
    value="None",
    description="Source:",
    layout=style_layout
)

tgt_lang_dropdown = Dropdown(
    options=lang_options,
    value="None",
    description="Target:",
    layout=style_layout
)

VBox([src_lang_dropdown, tgt_lang_dropdown])

In [ ]:
# @title Step 1 - Build models (takes a few minutes, but run only once)

import importlib
from auto_subtitle import Translator, build_whisper

print("Building Whisper model...")
proc, model, device_str = build_whisper("openai/whisper-large-v3", device=0)
print(f"Whisper ready on: {device_str}")

src_lang_name = src_lang_dropdown.value if "src_lang_dropdown" in globals() else "None"
tgt_lang_name = tgt_lang_dropdown.value if "tgt_lang_dropdown" in globals() else "None"
translation_enabled = src_lang_name != "None" and tgt_lang_name != "None" and src_lang_name != tgt_lang_name

translator = None
if translation_enabled:
    print(f"Building NLLB translator for {src_lang_name} -> {tgt_lang_name}...")
    translator = Translator()
    print(f"Translator ready on: {translator.get_device()}")
else:
    print("Translation is disabled.")

In [ ]:
# @title Step 2 - Provide link and transcribe
from pathlib import Path

from auto_subtitle import (
    LANGUAGE_CODES,
    Translator,
    download_file_if_possible,
    download_gdrive_file,
    extract_gdrive_file_id,
    transcribe_audio_file,
)

if "proc" not in globals() or "model" not in globals() or "device_str" not in globals():
    raise RuntimeError("Please run Step 1 first to build the Whisper model.")

file_link = input("Provide the Google Drive file link: ").strip()
if not file_link:
    raise ValueError("No link provided.")

print("Downloading the file...")
file_id = extract_gdrive_file_id(file_link)
print(f"Detected Google Drive File ID: {file_id}")
filename = download_gdrive_file(file_id=file_id, output=None)

input_file = Path(filename)
if not input_file.exists():
    raise RuntimeError("Download failed. Please check the link.")

output_srt = input_file.with_suffix(".srt")

src_lang_name = src_lang_dropdown.value if "src_lang_dropdown" in globals() else "None"
tgt_lang_name = tgt_lang_dropdown.value if "tgt_lang_dropdown" in globals() else "None"
translation_enabled = src_lang_name != "None" and tgt_lang_name != "None" and src_lang_name != tgt_lang_name

src_code = LANGUAGE_CODES.get(src_lang_name) if translation_enabled else None
tgt_code = LANGUAGE_CODES.get(tgt_lang_name) if translation_enabled else None

active_translator = None
if translation_enabled:
    if "translator" not in globals() or translator is None:
        print("Translator was not built in Step 1. Building it now...")
        translator = Translator()
    active_translator = translator
    print(f"Translation enabled: {src_lang_name} -> {tgt_lang_name}")
else:
    print("Translation disabled. Producing transcript only.")

transcribe_audio_file(
    proc=proc,
    model=model,
    device_str=device_str,
    input_path=input_file,
    out_srt=output_srt,
    batch_size=4,
    chunk_length_s=30,
    chunk_overlap_s=0.5,
    max_sub_duration=5.0,
    translator=active_translator,
    src_code=src_code,
    tgt_code=tgt_code,
)

print(f"SRT written to: {output_srt}")
download_file_if_possible(output_srt)